In [ ]:
#%%
fs = 1000 #Hz, frecuencia de muestreo

f0 = 1 #Hz, frecuencia de la senoidal

n=1000 # muestras por ciclo

vmax=1 #volt

dc=0 #valor medio, volts

ts = 1 / fs

def mi_funcion_sen(vmax, dc, f0,n, fs, ph=0):
    tt=np.linspace(0, (n-1)*ts,n)
    xx=dc+vmax*np.sin(2*np.pi*f0*tt+ph)
    
    return tt,xx

tt,xx=mi_funcion_sen(vmax, dc, f0,n, fs, ph=0)
#plt.subplot(2,1,1)
plt.plot(tt,xx)

#%%Señal con ruido
sigma=1
mu=0
U_n=np.random.normal(mu,sigma,n)
xxn=xx+U_n
#plt.subplot(2,1,2)
plt.plot(tt,xxn)

"""#%% f0=500 Hz
f0 = 500
tt, xx = mi_funcion_sen(vmax, dc, f0, n, fs, ph=0)
plt.figure()
plt.title('f0 = 500 Hz')
plt.plot(tt, xx)

#%% f0=999 Hz
f0 = 999
tt, xx = mi_funcion_sen(vmax, dc, f0, n, fs, ph=0)
plt.figure()
plt.title('f0 = 999 Hz')
plt.plot(tt, xx)

#%% f0=1001 Hz
f0 = 1001
tt, xx = mi_funcion_sen(vmax, dc, f0, n, fs, ph=0)
plt.figure()
plt.title('f0 = 1001 Hz')
plt.plot(tt, xx)

#%% f0=2001 Hz
f0 = 2001
tt, xx = mi_funcion_sen(vmax, dc, f0, n, fs, ph=0)
plt.figure()
plt.title('f0 = 2001 Hz')
plt.plot(tt, xx)

"""
#ph es fase
#niqwist
"""
#%%
"""#f0=35
#tt,xx=mi_funcion(vmax, dc, f0,n, fs, ph=np.pi/2)
#plt.plot(tt,xx)
#Potencia de la senoidal es A^2/2  #A es vmax
#Potencia del ruido es sigma^2
"""
"""
#%% Potencia de señal
#potencia es **
vmax=np.sqrt(2)
tt,xx=mi_funcion_sen(vmax, dc, f0,n, fs, ph=0)
Px=np.var(xx)
SNR=20
Pr=10**(-SNR/10) #para llegar al resultado hay que usar ln.
U_n = np.random.normal(mu,np.sqrt(Pr),n)

xxn=xx+U_n

plt.figure(1)
plt.clf()
plt.plot(tt,xxn)
plt.plot(tt, xx, 'r', lw=2 )
plt.title(f'f0={f0} Hz')

#%%

from scipy import signal as sig

n0= 300 #muestras

dd= np.zeros(n0+1)
dd[n0]=1.

yy=sig.convolve(xx, dd)

plt.figure(2)
plt.clf()
plt.plot(yy)

#%%
#Con -1 veo la ultima muestra de un vector y me sirve para ver que se flipea.

yy=1/n*sig.convolve(U_n, np.flip(U_n))

plt.figure(3)
plt.clf()
plt.plot(yy)

#%% CUANTIZACION

B = 4 # Cantidad de bits
Vfs = 3 # [V] --> es el voltaje en full scale
# --> asumo que es simetrico, en este caso como es 3 tengo 1.5 para arriba y 1.5 para abajo
qq = Vfs / 2**B# Rango de cuantizacion --> qq = 0,1875

# Para cuantizar tengo que truncar o redondear
xxq = np.round(xx / qq) * qq # Esta cuantizada y codificada
# Lo multiplico devuelta por qq para que me coincida con las unidades de xx

plt.figure(figsize=(8,6))
plt.subplot(2,1,1)
plt.plot(xx, label='Señal original')
plt.plot(xxq, label='Señal cuantizada')
plt.grid(True)
plt.legend()
plt.title("Cuantización")

# Calculo la secuencia de error
# El error de cuntizacion queda acotado entre [-q/2; +q/2] --> en este caso [-0,1875; +0,1875]
# Para calcular el error xx y xxq tienen que estar en la misma magnitud, en lass mismas unidades
# --> Entonces multiplico devuelta por q, no se va a simplificar porque es una operacion no lineal
e = xxq - xx # Es el error de cuantizacion

# Grafico el error 
plt.subplot(2,1,2)
plt.plot(tt, e, label = 'Error de cuantizacion')
plt.axhline( qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'+q/2 = +{qq/2:.4f}V')
plt.axhline(-qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'-q/2 = -{qq/2:.4f}V')
plt.legend()
plt.grid(True)

plt.xlabel("Muestras")
plt.ylabel("Amplitud [V]")

plt.tight_layout()
plt.show()

# --> En este caso tenemos algo correlado
# Hay mucho ordenamiento temporal 

#%% Ahora cuantizamos xxn que tiene un poco de ruido (20db de SNR)

xxq = np.round(xxn / qq) * qq

plt.figure()
plt.subplot(2,1,1)
plt.plot(xxn, label = 'Señal original con ruido')
plt.plot(xxq, label = 'Señal cuantizada')
plt.grid(True)
plt.legend()
plt.title("Cuantización con ruido")

# Calculo la secuencia de error
# El error de cuntizacion queda acotado entre [-q/2; +q/2] --> en este caso [-0,1875; +0,1875]
# Para calcular el error xx y xxq tienen que estar en la misma magnitud, en lass mismas unidades
# --> Entonces multiplico devuelta por q, no se va a simplificar porque es una operacion no lineal
e = xxq - xxn # Es el error de cuantizacion

# Grafico el error 
plt.subplot(2,1,2)
plt.plot(tt, e, label = 'Error de cuantizacion')
plt.axhline( qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'+q/2 = +{qq/2:.4f}V')
plt.axhline(-qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'-q/2 = -{qq/2:.4f}V')
plt.legend()
plt.grid(True)

plt.xlabel("Muestras")
plt.ylabel("Amplitud [V]")

plt.tight_layout()
plt.show()

# --> En este caso tenemos algo fuertemente incorrelado, que tiene que ver con la potencia del ruido
# Es decir que se "rompe" la correlacion --> eso es porque cambiamos Pr (la potencia), entonces hay mas energia, y hay exceso de ruido
# --> Tiene que ver con la cantidad de bits
# Cuando llego a los limites de cuantizacion, hace que se generen cambios de bits, cambios de niveles
# --> Para romper con eso tiene que ser comparable con la mitad del paso de cuantizacion

# La potencia del ruido es tan grande que hace que cuando lo queres representar se pasa al nivel siguiente de cuantizacion

#%% SEÑAL SIN RUIDO 

e = xxq - xx

# HISTOGRAMA
# --> es la funcion distribucion estocastica, es un estimador de la funcion distribucion de probabilidad
# Es una aproximacion a la funcion teorica (cajita) 
plt.figure(figsize=(10,6))

plt.subplot(1,2,1)
plt.hist(e, bins = 30, color = 'steelblue', density = True, edgecolor = 'dimgray', linewidth = 0.8) # La cantidad de bins me da mas rayitas, las cajitas son mas angostas. Si es muy angosta me va a caer en cero porque es muy angostita la cajita
plt.axvline( qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'+q/2 = +{qq/2:.4f}V')
plt.axvline(-qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'-q/2 = -{qq/2:.4f}V')
plt.axhline(1/qq, color='darkorange',   linewidth=0.8, linestyle='--', label=f'1/q = {1/qq:.4f}V')
plt.grid(True)
plt.legend()
plt.title("Histograma del error de cuantización (sin ruido)")
plt.xlabel("Error [V]")
plt.ylabel("Densidad")

# AUTOCORRELACION
plt.subplot(1,2,2)

r = np.correlate(e, e, mode = 'full')
lags = np.arange(-len(e)+1, len(e))
L = 50 # Cantidad de lags que quiero ver

plt.plot(lags, r, label = 'Autocorrelacion', color = 'seagreen')
plt.xlim(-L, L)
plt.axvline(0, color='red',   linewidth=0.8, linestyle='--', label='Lag = 0')
plt.legend()
plt.grid(True)
plt.title("Autocorrelación del error (sin rudio)")
plt.xlabel("Retardo (lag)")
plt.ylabel("Autocorrelación")

#%% SEÑAL CON RUIDO 

e = xxq - xxn

# HISTOGRAMA 
plt.figure(figsize=(10,6))

plt.subplot(1,2,1)
plt.hist(e, bins = 30, color = 'steelblue', density = True, edgecolor = 'dimgray', linewidth = 0.8) # La cantidad de bins me da mas rayitas, las cajitas son mas angostas. Si es muy angosta me va a caer en cero porque es muy angostita la cajita
plt.axvline( qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'+q/2 = +{qq/2:.4f}V')
plt.axvline(-qq/2, color='red',   linewidth=0.8, linestyle='--', label=f'-q/2 = -{qq/2:.4f}V')
plt.axhline(1/qq, color='darkorange',   linewidth=0.8, linestyle='--', label=f'1/q = {1/qq:.4f}V')
plt.grid(True)
plt.legend()
plt.title("Histograma del error de cuantización (con ruido)")
plt.xlabel("Error [V]")
plt.ylabel("Densidad")

# AUTOCORRELACION
plt.subplot(1,2,2)

r = np.correlate(e, e, mode = 'full')
lags = np.arange(-len(e)+1, len(e))
L = 50 # Cantidad de lags que quiero ver

plt.plot(lags, r, label = 'Autocorrelacion', color = 'seagreen')
plt.xlim(-L, L)
plt.axvline(0, color='red',   linewidth=0.8, linestyle='--', label='Lag = 0')
plt.legend()
plt.grid(True)
plt.title("Autocorrelación del error (con rudio)")
plt.xlabel("Retardo (lag)")
plt.ylabel("Autocorrelación")

#%% FFT
# Le pasamos secuencias reales, y me va a devolver secuencias complejas (modulo y fase)

XX = np.fft.fft(xx)

XXmod = np.abs(XX) # MODULO
XXfase = np.angle(XX) # FASE --> arcotangente de la parte real sobre la parte imaginaria --> va de -pi a pi --> hago un arcotangente de cuatro cuadrantes (lo comun es de dos cuadrantes)

plt.figure(figsize=(10,6))
plt.subplot(1,2,1)
plt.plot(XXmod)
plt.title('Modulo de la DFT de xx')
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(XXfase)
plt.title('Fase de la DFT de xx')
plt.grid(True)

plt.tight_layout()
plt.show()